# Setup


In [ ]:
%pip install -q -U "crewai[google-genai,tools]"

# Auth & Model


In [ ]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM
from pydantic import BaseModel, Field

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Please add GEMINI_API_KEY to Google Colab Secrets.")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    temperature=0.2
)

print("Gemini configured successfully!")

# Sales data


In [ ]:
import pandas as pd

sales_data = {
    "Date": [
        "2026-10-01", "2026-10-02", "2026-10-03",
        "2026-10-04", "2026-10-05", "2026-10-06",
        "2026-10-07", "2026-10-08", "2026-10-09",
        "2026-10-10"
    ],
    "Product": [
        "Workstation", "Smartwatch", "Earbuds", "Workstation", "Monitor",
        "Smartwatch", "Earbuds", "Monitor", "Workstation", "Smartwatch"
    ],
    "Region": [
        "South", "North", "East", "West", "South",
        "North", "East", "West", "South", "North"
    ],
    "Units_Sold": [6, 14, 22, 5, 9, 11, 28, 8, 7, 16],
    "Unit_Price": [62000, 18000, 3500, 62000, 24000,
                   18000, 3500, 24000, 62000, 18000]
}

df = pd.DataFrame(sales_data)
df["Revenue"] = df["Units_Sold"] * df["Unit_Price"]

df.to_csv("sales.csv", index=False)

display(df)
print("sales.csv created successfully!")

# File tools


In [ ]:
from crewai_tools import FileReadTool, FileWriterTool

file_reader = FileReadTool()
file_writer = FileWriterTool()

print("File tools initialized successfully!")

# Schema


In [ ]:
class SalesReport(BaseModel):
    total_records: int = Field(description="Number of sales records")
    total_units_sold: int = Field(description="Total units sold")
    total_revenue: float = Field(description="Total revenue across all records")
    best_selling_product: str = Field(description="Product with the highest total units sold")
    highest_revenue_region: str = Field(description="Region with the highest total revenue")
    average_revenue_per_record: float = Field(description="Average revenue per sales record")

# Analyst agent


In [ ]:
data_analyst = Agent(
    role="Commercial Data Analyst",
    goal="Analyze sales data accurately and produce structured sales statistics.",
    backstory=(
        "You are a careful data analyst who examines sales records, "
        "calculates business metrics, and identifies product and regional trends. "
        "Use the supplied CSV data and never invent figures."
    ),
    llm=llm,
    tools=[file_reader],
    verbose=True
)

# Analysis task


In [ ]:

analysis_task = Task(
    description="""
    Analyze the sales data stored in sales.csv.

    Calculate the following metrics:
    1. Total number of sales records.
    2. Total units sold across all products.
    3. Total revenue across all records.
    4. Best-selling product based on total units sold.
    5. Region with the highest total revenue.
    6. Average revenue per sales record.

    Instructions:
    - Read the CSV file using the available file-reading tool.
    - Use the actual data to calculate every metric.
    - Group records by product to identify the best-selling product.
    - Group records by region to identify the highest-revenue region.
    - Do not guess or invent values.
    - Return the results in the required structured format.
    """,
    expected_output=(
        "A structured sales analysis containing all six metrics, "
        "with accurate values derived from sales.csv."
    ),
    agent=data_analyst,
    output_pydantic=SalesReport
)
print("Analysis task created successfully!")


# Insight agent


In [ ]:

insight_writer = Agent(
    role="Commercial Insight Specialist",
    goal=(
        "Convert sales statistics into clear, useful business "
        "insights and recommendations."
    ),
    backstory=(
        "You are a business analyst who explains sales trends "
        "in simple language and provides practical recommendations "
        "based only on the available data."
    ),
    llm=llm,
    verbose=True
)

print("Insight Writer agent created successfully!")


# Reviewer agent


In [ ]:

reviewer = Agent(
    role="Report Quality Reviewer",
    goal=(
        "Review the sales report for numerical accuracy, "
        "clarity, consistency, and actionable recommendations."
    ),
    backstory=(
        "You are a meticulous report reviewer. You verify claims "
        "against the supplied analysis, flag unsupported conclusions, "
        "and improve the report's readability."
    ),
    llm=llm,
    verbose=True
)

print("Reviewer agent created successfully!")


# Insight task


In [ ]:

insight_task = Task(
    description="""
    Using the sales analysis provided in the task context:

    1. Explain the overall sales performance.
    2. Identify the strongest-selling product.
    3. Explain which region generated the most revenue.
    4. Discuss what the average revenue per record indicates.
    5. Provide three practical business recommendations.

    Base every numerical claim on the analysis.
    Do not invent trends or claim growth over time because the
    dataset covers only a small sample of records.
    Write the report in clear Markdown with headings and bullet points.
    """,
    expected_output=(
        "A Markdown sales report containing a summary, key findings, "
        "and three data-supported business recommendations."
    ),
    agent=insight_writer,
    context=[analysis_task]
)

print("Insight writing task created successfully!")


# Review task


In [ ]:

review_task = Task(
    description="""
    Review the draft sales report using the original structured
    sales analysis available in the task context.

    Check:
    1. Whether numerical claims match the analysis.
    2. Whether product and regional findings are consistent.
    3. Whether recommendations follow from the available evidence.
    4. Whether the report is clear, well-organized, and professional.

    Correct errors and return the complete revised report in Markdown.
    Do not invent additional statistics.
    """,
    expected_output=(
        "A reviewed, corrected, professional Markdown sales report."
    ),
    agent=reviewer,
    context=[analysis_task, insight_task],
    output_file="report.md"
)

print("Review task created successfully!")


# Crew


In [ ]:

sales_crew = Crew(
    agents=[
        data_analyst,
        insight_writer,
        reviewer
    ],
    tasks=[
        analysis_task,
        insight_task,
        review_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Data Report Crew assembled successfully!")


# Run


In [ ]:

result = await sales_crew.kickoff_async()

print("\n===== FINAL SALES REPORT =====\n")
print(result)

print("\nCheck the Colab Files panel for report.md")


# Verified metrics


In [ ]:

verified_metrics = {
    "total_records": int(len(df)),
    "total_units_sold": int(df["Units_Sold"].sum()),
    "total_revenue": int(df["Revenue"].sum()),
    "best_selling_product": str(
        df.groupby("Product")["Units_Sold"].sum().idxmax()
    ),
    "best_selling_product_units": int(
        df.groupby("Product")["Units_Sold"].sum().max()
    ),
    "highest_revenue_product": str(
        df.groupby("Product")["Revenue"].sum().idxmax()
    ),
    "highest_revenue_region": str(
        df.groupby("Region")["Revenue"].sum().idxmax()
    ),
    "highest_revenue_region_total": int(
        df.groupby("Region")["Revenue"].sum().max()
    ),
    "average_revenue_per_record": float(
        df["Revenue"].mean()
    )
}

print(verified_metrics)


# Update task


In [ ]:

insight_task.description = f"""
Write a professional sales performance report using these
verified metrics calculated directly from sales.csv:

{verified_metrics}

Include the following sections:

1. Overall Sales Performance
   - Total records, units sold, total revenue, and average
     revenue per record.

2. Product Performance
   - Identify the best-selling product by units sold.
   - Identify the product generating the highest revenue.
   - Clearly distinguish units sold from revenue generated.

3. Regional Performance
   - Identify the region with the highest revenue.
   - Include its verified revenue figure.

4. Business Recommendations
   - Provide three practical recommendations supported by
     the available data.

Rules:
- Use only the verified numerical values supplied above.
- Do not recalculate, modify, or invent statistics.
- Use ₹ as the currency symbol.
- Do not claim sales growth or trends over time.
- Format the report using clean Markdown headings and bullets.
"""

print("Insight Writer task updated successfully!")


# Update review


In [ ]:

review_task.description = f"""
Review the draft sales report against these verified metrics:

{verified_metrics}

Check that:
1. Total records, units sold, total revenue, and average revenue
   match the verified values.
2. The best-selling product by units is correctly identified.
3. The highest-revenue product is correctly identified.
4. The highest-revenue region and its revenue are correct.
5. Every numerical claim is supported by the verified metrics.
6. Recommendations are reasonable and do not claim unsupported
   sales growth or customer behavior.

Correct every discrepancy and return the complete final report
in Markdown.

Use ₹ for currency. Do not invent figures or alter verified values.
"""

print("Reviewer task updated successfully!")


# Output


In [ ]:

import os

report_path = "report.md"

if os.path.exists(report_path):
    print("Report saved successfully!")
    print("File size:", os.path.getsize(report_path), "bytes")

    with open(report_path, "r", encoding="utf-8") as file:
        print("\n===== report.md =====\n")
        print(file.read())
else:
    print("report.md was not found in the current directory.")
    print("Check the Colab Files panel.")
